In [18]:
import numpy as np

TEXT = 'You say goodbye and I say hello.'

In [33]:
def proprecess(text: str):
    '''
    proprecess
    '''
    text = text.lower()
    text = text.replace('.', ' .')
    words = text.split(' ')

    word_to_id = {}
    id_to_word = {}

    for word in words:
        if word not in word_to_id:
            new_id = len(word_to_id)
            word_to_id[word] = new_id
            id_to_word[new_id] = word

    corpus = np.array([word_to_id[w] for w in words])

    return corpus, word_to_id, id_to_word

corpus, word_to_id, id_to_word = proprecess(TEXT)

print(corpus, word_to_id, id_to_word)


[0 1 2 3 4 1 5 6] {'you': 0, 'say': 1, 'goodbye': 2, 'and': 3, 'i': 4, 'hello': 5, '.': 6} {0: 'you', 1: 'say', 2: 'goodbye', 3: 'and', 4: 'i', 5: 'hello', 6: '.'}


In [34]:
def create_co_matrix(corpus, vocab_size, window_size = 1):
    '''
    create comatrix
    '''
    corpus_size = len(corpus)
    corpus_matrix = np.zeros(shape=(vocab_size, vocab_size), dtype=np.int32)

    for index, word_id in enumerate(corpus):
        for i in range(1, window_size + 1):
            left_index = index - i
            right_index = index + i

            if left_index >= 0:
                left_word_id = corpus[left_index]
                corpus_matrix[word_id, left_word_id] += 1
            
            if right_index < corpus_size:
                right_word_id = corpus[right_index]
                corpus_matrix[word_id, right_word_id] += 1
    return corpus_matrix

In [35]:
def cos_similarity(x, y, eps = 1e-8):
    '''
    cos similarity
    '''
    nx = x / np.sqrt(np.sum(x ** 2) + eps) # x 正规化
    ny = y / np.sqrt(np.sum(y ** 2) + eps) # y 正规化
    # eps 防止分母为零
    return np.dot(nx, ny)

In [36]:
corpus, word_to_id, id_to_word = proprecess(TEXT)
print(corpus, word_to_id, id_to_word)
C = create_co_matrix(corpus, len(word_to_id), window_size=1)
c0 = C[word_to_id['you']]
c1 = C[word_to_id['i']]

similarity = cos_similarity(c0, c1)
print(similarity)

[0 1 2 3 4 1 5 6] {'you': 0, 'say': 1, 'goodbye': 2, 'and': 3, 'i': 4, 'hello': 5, '.': 6} {0: 'you', 1: 'say', 2: 'goodbye', 3: 'and', 4: 'i', 5: 'hello', 6: '.'}
0.7071067758832467


In [ ]:
def most_similar(query, word_to_id, id_to_word, word_matrix, top=5):
    '''
    most similar
    '''
    if query not in word_to_id:
        print(f"{query} is not found")
        return

    print(f"\n[query] {query}")
    query_id = word_to_id[query]
    query_vec = word_matrix[query_id]

    vocab_size = len(id_to_word)
    similarity = np.zeros(vocab_size)

    for i in range(vocab_size):
        similarity[i] = cos_similarity(word_matrix[i], query_vec)

    count = 0
    for i in (-1 * similarity).argsort():
        if id_to_word[i] == query:
            continue
        print(f"{id_to_word[i]}: {similarity[i]}")

        count += 1
        if count >= top:
            return